In [1]:
from wrapper.env_evaluator import RobosuiteEvaluator

from robokit.debug_utils.printer import print_batch
from robokit.debug_utils.images import save_frames_as_video

In [3]:
robosuite_env = RobosuiteEvaluator(
    env_name="Stack",
    use_camera_obs=True,
)

In [4]:
state, info_dict = robosuite_env.reset()
print_batch("info_dict", info_dict)

info_dict: Dict, keys=['obs', 'agent_state', 'env_state', 'success', 'step_count']
--obs: Dict, keys=['robot0_joint_pos_cos', 'robot0_joint_pos_sin', 'robot0_joint_vel', 'robot0_eef_pos', 'robot0_eef_quat', 'robot0_gripper_qpos', 'robot0_gripper_qvel', 'agentview_image', 'sideview_image', 'cubeA_pos', 'cubeA_quat', 'cubeB_pos', 'cubeB_quat', 'gripper_to_cubeA', 'gripper_to_cubeB', 'cubeA_to_cubeB', 'robot0_proprio-state', 'object-state']
----robot0_joint_pos_cos, <class 'numpy.ndarray'>, shape=(7,), min=-0.3529, max=1.0000, dtype=float64
----robot0_joint_pos_sin, <class 'numpy.ndarray'>, shape=(7,), min=-0.9998, max=0.9357, dtype=float64
----robot0_joint_vel, <class 'numpy.ndarray'>, shape=(7,), min=0.0000, max=0.0000, dtype=float64
----robot0_eef_pos, <class 'numpy.ndarray'>, shape=(3,), min=-0.1567, max=0.9398, dtype=float64
----robot0_eef_quat, <class 'numpy.ndarray'>, shape=(4,), min=-0.9995, max=0.0313, dtype=float64
----robot0_gripper_qpos, <class 'numpy.ndarray'>, shape=(6,), mi

In [9]:
import os
import numpy as np

# Block 1: 随机 action rollout
# action shape: (A_policy,), A_policy = robosuite_env.policy_action_dim
# action range: [policy_action_low, policy_action_high]

robosuite_env = RobosuiteEvaluator(
    env_name="TwoArmPegInHole",
    use_camera_obs=True,
)

num_random_steps = 50
root_dir = "/mnt/dongxu-fs1/data-hdd/geyuan/code/trdrl25nips/robosuite/"
video_save_path = f"{root_dir}/debug/videos/{robosuite_env.env_name}_random_rollout.mp4"
video_fps = 30

state, info = robosuite_env.reset()

random_rollout = []
random_frames = []

init_frame = robosuite_env.get_frame(info.get("obs"))
if init_frame is not None:
    random_frames.append(init_frame)

for t in range(num_random_steps):
    action = np.random.uniform(
        low=robosuite_env.policy_action_low,
        high=robosuite_env.policy_action_high,
        size=(robosuite_env.policy_action_dim,),
    ).astype(np.float32)

    next_state, reward, done, step_info = robosuite_env.step(action)

    frame = robosuite_env.get_frame(step_info.get("obs"))
    if frame is not None:
        random_frames.append(frame)

    random_rollout.append({
        "t": t,
        "reward": reward,
        "done": done,
        "success": step_info.get("success", 0),
    })

    state = next_state
    if done:
        break

if len(random_frames) > 0:
    save_frames_as_video(random_frames, video_save_path, fps=video_fps)
    print(f"Saved random rollout video: {video_save_path}, frames={len(random_frames)}")
else:
    print("No valid camera frames found, skip video saving.")

print(f"Random rollout finished: steps={len(random_rollout)}, total_reward={sum(x['reward'] for x in random_rollout):.3f}")
random_rollout[-5:]


next_state, <class 'numpy.ndarray'>, shape=(26,), min=-0.9993, max=1.1118, dtype=float64
Saved random rollout video: /mnt/dongxu-fs1/data-hdd/geyuan/code/trdrl25nips/robosuite//debug/videos/TwoArmPegInHole_random_rollout.mp4, frames=51
Random rollout finished: steps=50, total_reward=0.000


[{'t': 45, 'reward': 0.0, 'done': False, 'success': 0},
 {'t': 46, 'reward': 0.0, 'done': False, 'success': 0},
 {'t': 47, 'reward': 0.0, 'done': False, 'success': 0},
 {'t': 48, 'reward': 0.0, 'done': False, 'success': 0},
 {'t': 49, 'reward': 0.0, 'done': False, 'success': 0}]

In [2]:
import os
import numpy as np

# Block 2: 使用预收集 demo 的 action rollout（自动遍历所有 demo episode）
# 默认 demo 文件格式: [state, action, reward, next_state, done]
# transitions shape: (N, 5), action shape: (A_env,) 或 (A_policy,)

robosuite_env = RobosuiteEvaluator(
    # env_name="Stack",
    env_name="NutAssemblyRound",
    # env_name="TwoArmPegInHole",
    use_camera_obs=True,
)

root_dir = "/mnt/dongxu-fs1/data-hdd/geyuan/code/trdrl25nips/robosuite/"
demo_transition_path = f"{root_dir}/generate/{robosuite_env.env_name}_transitions_10trajectory_sparse.npy"
assert os.path.exists(demo_transition_path), f"Demo file not found: {demo_transition_path}"

transitions = np.load(demo_transition_path, allow_pickle=True)
all_states = np.stack(transitions[:, 0], axis=0).astype(np.float32)
all_actions = np.stack(transitions[:, 1]).astype(np.float32)
print_batch("all_states", all_states)

# 如果 demo action 是 env action（例如 TwoArm 可能是 A_env），转换到 evaluator 的 policy action 维度
if all_actions.shape[1] == robosuite_env.env_action_dim and robosuite_env.policy_action_dim < robosuite_env.env_action_dim:
    all_policy_actions = all_actions[:, : robosuite_env.policy_action_dim]
elif all_actions.shape[1] == robosuite_env.policy_action_dim:
    all_policy_actions = all_actions
else:
    raise ValueError(
        f"Unexpected action dim in demo: {all_actions.shape[1]}, "
        f"expected {robosuite_env.policy_action_dim} or {robosuite_env.env_action_dim}"
    )

# 按 env horizon 切分 episode，和 reset_to_demo_start 的 episode 索引保持一致
episode_horizon = int(robosuite_env.horizon)
num_episodes = int(np.ceil(len(all_policy_actions) / episode_horizon))

# 视频保存设置
save_replay_videos = True
video_fps = 30
video_save_dir = f"{root_dir}/debug/videos/{robosuite_env.env_name}_demo_replay_all"
if save_replay_videos:
    os.makedirs(video_save_dir, exist_ok=True)

replay_stats = []
success_cnt = 0

for episode_idx in range(num_episodes):
    start = episode_idx * episode_horizon
    end = min((episode_idx + 1) * episode_horizon, len(all_policy_actions))
    if start >= end:
        continue

    # 将环境 reset 到对应 demo episode 的起点
    state, info = robosuite_env.reset_to_demo_start(
        transitions=transitions,
        episode_idx=episode_idx,
        horizon=episode_horizon,
    )

    episode_success = False
    episode_reward = 0.0
    executed_steps = 0
    episode_frames = []

    init_frame = robosuite_env.get_frame(info.get("obs"))
    if init_frame is not None:
        episode_frames.append(init_frame)

    for action in all_policy_actions[start:end]:
        next_state, reward, done, step_info = robosuite_env.step(action)

        episode_reward += float(reward)
        executed_steps += 1
        if bool(step_info.get("success", 0)):
            episode_success = True

        frame = robosuite_env.get_frame(step_info.get("obs"))
        if frame is not None:
            episode_frames.append(frame)

        state = next_state
        if done:
            break

    video_path = None
    if save_replay_videos and len(episode_frames) > 0:
        video_path = (
            f"{video_save_dir}/ep{episode_idx:03d}_"
            f"suc{int(episode_success)}_steps{executed_steps}.mp4"
        )
        save_frames_as_video(episode_frames, video_path, fps=video_fps)

    success_cnt += int(episode_success)
    replay_stats.append({
        "episode_idx": episode_idx,
        "start": start,
        "end": end,
        "steps": executed_steps,
        "success": episode_success,
        "total_reward": episode_reward,
        "video_path": video_path,
    })

    print(
        f"[Replay] ep={episode_idx:02d}/{num_episodes - 1:02d}, "
        f"success={int(episode_success)}, steps={executed_steps}, "
        f"total_reward={episode_reward:.3f}, video={video_path}"
    )

print(
    f"Replay summary: success={success_cnt}/{len(replay_stats)} "
    f"({100.0 * success_cnt / max(len(replay_stats), 1):.2f}%), "
    f"demo_path={demo_transition_path}, video_dir={video_save_dir}"
)

replay_stats


all_states, <class 'numpy.ndarray'>, shape=(5000, 22), min=-0.3663, max=1.0000, dtype=float32
[DEBUG] reset to demo_start: self._state: [-0.1624859  -0.02484444  0.81956511 -0.10111389 -0.12518343  0.88999999
  0.1        -0.1         0.85        0.1        -0.1         0.85000002
 -0.06137201  0.100339   -0.07043488 -0.20111389 -0.02518343  0.03999999
 -0.20111389 -0.02518343  0.03999996  0.        ], reset_match_l2: 0.20514508192471853
[Replay] ep=00/09, success=0, steps=500, total_reward=0.000, video=/mnt/dongxu-fs1/data-hdd/geyuan/code/trdrl25nips/robosuite//debug/videos/NutAssemblyRound_demo_replay_all/ep000_suc0_steps500.mp4
[DEBUG] reset to demo_start: self._state: [-0.1530593  -0.02484429  0.80642827 -0.1980633  -0.13211645  0.88999999
  0.1        -0.1         0.85        0.1        -0.1         0.85000002
  0.045004    0.10727217 -0.08357171 -0.2980633  -0.03211645  0.03999999
 -0.2980633  -0.03211645  0.03999996  0.        ], reset_match_l2: 0.2051671177931095
[Replay] ep=01

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7fb96206c520>>
Traceback (most recent call last):
  File "/mnt/dongxu-fs1/data-ssd/geyuan/programs/anaconda3/envs/trdrl/lib/python3.8/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


[DEBUG] reset to demo_start: self._state: [-0.16100727 -0.02484432  0.80975312 -0.14074881 -0.16877301  0.88999999
  0.1        -0.1         0.85        0.1        -0.1         0.85000002
 -0.02025846  0.14392869 -0.08024686 -0.24074881 -0.06877301  0.03999999
 -0.24074882 -0.06877301  0.03999996  0.        ], reset_match_l2: 0.2053331232394909
[Replay] ep=05/09, success=0, steps=500, total_reward=0.000, video=/mnt/dongxu-fs1/data-hdd/geyuan/code/trdrl25nips/robosuite//debug/videos/NutAssemblyRound_demo_replay_all/ep005_suc0_steps500.mp4
[DEBUG] reset to demo_start: self._state: [-0.1711436  -0.02484443  0.81853729 -0.18565769 -0.19216131  0.88999999
  0.1        -0.1         0.85        0.1        -0.1         0.85000002
  0.0145141   0.16731688 -0.0714627  -0.28565769 -0.09216131  0.03999999
 -0.2856577  -0.09216131  0.03999996  0.        ], reset_match_l2: 0.20525993088587607
[Replay] ep=06/09, success=0, steps=500, total_reward=0.000, video=/mnt/dongxu-fs1/data-hdd/geyuan/code/trdr

[{'episode_idx': 0,
  'start': 0,
  'end': 500,
  'steps': 500,
  'success': False,
  'total_reward': 0.0,
  'video_path': '/mnt/dongxu-fs1/data-hdd/geyuan/code/trdrl25nips/robosuite//debug/videos/NutAssemblyRound_demo_replay_all/ep000_suc0_steps500.mp4'},
 {'episode_idx': 1,
  'start': 500,
  'end': 1000,
  'steps': 500,
  'success': False,
  'total_reward': 0.0,
  'video_path': '/mnt/dongxu-fs1/data-hdd/geyuan/code/trdrl25nips/robosuite//debug/videos/NutAssemblyRound_demo_replay_all/ep001_suc0_steps500.mp4'},
 {'episode_idx': 2,
  'start': 1000,
  'end': 1500,
  'steps': 500,
  'success': False,
  'total_reward': 0.0,
  'video_path': '/mnt/dongxu-fs1/data-hdd/geyuan/code/trdrl25nips/robosuite//debug/videos/NutAssemblyRound_demo_replay_all/ep002_suc0_steps500.mp4'},
 {'episode_idx': 3,
  'start': 1500,
  'end': 2000,
  'steps': 500,
  'success': False,
  'total_reward': 0.0,
  'video_path': '/mnt/dongxu-fs1/data-hdd/geyuan/code/trdrl25nips/robosuite//debug/videos/NutAssemblyRound_demo